In [ ]:
# https://geojson.io/
# https://www.cenenekretnina.rs/
# https://katastar.rgz.gov.rs/RegistarCenaNepokretnosti/
# https://t.me/PokupkaKvartiriSerbii

In [ ]:
from tqdm import tqdm
import requests
import json

# with open("db.json", "r") as file:
#     db = json.load(file)

def update_cityexpert(cities=[1, 2, 3]):
    for city in cities:
        for rentOrSale, key in [("r", "rent"), ("s", "sale")]:
            # req = {"cityId": city, "rentOrSale": rentOrSale, "searchSource": "regular", "sort": "datedsc"}
            # properties = requests.get("https://cityexpert.rs/api/Search/Map?req=" + json.dumps(req)).json()
            # ids = [p["propId"] for p in properties if str(p["propId"]) not in db["cityexpert"][key]]

            ids = [id for id in range(int(max(db["cityexpert"][key])), 0, -1) if str(id) not in db["cityexpert"][key]]

            for id in tqdm(ids):
                text = requests.get(f"https://cityexpert.rs/api/PropertyView/{id}/{rentOrSale}").text

                try:
                    db["cityexpert"][key][id] = json.loads(text)
                except Exception as e:
                    db["cityexpert"][key][id] = text

update_cityexpert([1])

# with open("db.json", "w") as file:
#     json.dump(db, file, indent=2, ensure_ascii=False)

In [ ]:
with open("db.json", "w") as file:
    json.dump(db, file, indent=2, ensure_ascii=False)

In [ ]:
from sklearn.linear_model import LinearRegression
import branca.colormap as cm
import pandas as pd
import folium
import json

with open("db.json", "r") as file:
    db = json.load(file)

city = 1

rent_df = pd.DataFrame([item for item in db["cityexpert"]["rent"].values() if type(item) == dict and item["cityId"] == city])
rent_model = LinearRegression()
rent_model.fit(rent_df[["mapLat", "mapLng", "size"]], rent_df["price"] / rent_df["size"])

sale_df = pd.DataFrame([item for item in db["cityexpert"]["sale"].values() if type(item) == dict and item["cityId"] == city])
sale_df["rent_per_size"] = rent_model.predict(sale_df[["mapLat", "mapLng", "size"]])

colormap = cm.linear.YlOrRd_09.scale(0, 30)
map = folium.Map(location=[sale_df["mapLat"].mean(), sale_df["mapLng"].mean()], zoom_start=12, tiles="OpenStreetMap")

for _, row in sale_df.iterrows():
    rent_mo = row["rent_per_size"] * row["size"]
    ratio = row["price"] / (12 * rent_mo)

    folium.CircleMarker(
        location=[row["mapLat"], row["mapLng"]], radius=5, fill=True, fill_opacity=1, color=colormap(ratio),
        tooltip=f'{100 * int(row["price"] / 100):,}€ | {int(rent_mo / 10) * 10:,}€ | {int(row["size"])}m² | {ratio:.1f}').add_to(map)

colormap.add_to(map)
map

In [ ]:
import branca.colormap as cm
from rtree import index
import folium
import json

with open("db.json", "r") as file:
    db = json.load(file)

def read_type(type_item):
    df = pd.DataFrame([v for v in db["cityexpert"][type_item].values() if type(v) == dict])
    df = df[["mapLat", "mapLng", "size", "price"]]
    df = df.rename(columns={"mapLat": "lat", "mapLng": "lon", "size": "sq_m"})
    df["sq_m_price"] = df["price"] / df["sq_m"]
    return df

rent_df, sale_df = read_type("rent"), read_type("sale")
rent_idx = index.Index()

rent_df = rent_df[rent_df["sq_m"] < 70]
sale_df = sale_df[sale_df["sq_m"] < 70]

for i, row in rent_df.iterrows():
    rent_idx.insert(i, (row["lat"], row["lon"], row["lat"], row["lon"]))

def get_neighbours(lat, lon, diff=0.005):
    return rent_df.loc[rent_idx.intersection((lat - diff, lon - diff, lat + diff, lon + diff))]

colormap = cm.linear.YlOrRd_09.scale(0, 30)
map = folium.Map(location=[44.795, 20.458], zoom_start=12, tiles="OpenStreetMap")

for _, row in sale_df.iterrows():
    neighbours = get_neighbours(row["lat"], row["lon"])

    if len(neighbours) < 3:
        continue
    
    rent_mo = neighbours["sq_m_price"].mean() * row["sq_m"]
    ratio = row["price"] / (12 * rent_mo)

    stats = {
        "neighbours": len(neighbours),
        "price": f'{100 * int(row["price"] / 100):,}€',
        "rent": f'{int(rent_mo / 10) * 10:,}€',
        "sq_m": f'{int(row["sq_m"])}m²',
        "ratio": f'{ratio:.1f}',
        "sq_m_std": f'{neighbours["sq_m"].std() / row["sq_m"]:.3f}',
        "price_std": f'{neighbours["price"].std() / rent_mo:.3f}'
    }

    folium.CircleMarker(
        location=[row["lat"], row["lon"]], radius=5, fill=True, fill_opacity=1,
        color=colormap(ratio),
        tooltip=" | ".join([f"{k}: {v}" for k, v in stats.items()])).add_to(map)

colormap.add_to(map)
map

In [102]:
import requests
import json

data = requests.post(
    "https://katastar.rgz.gov.rs/RegistarCenaNepokretnosti/Default.aspx/Data",
    headers={
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"    
    }, data=json.dumps({
        "DatumPocetak": "01.01.2023",
        "DatumZavrsetak": "01.01.2025",
        "OpstinaID": "70114",
        "KoID": "-1",
        "VrsteNepokretnosti": "831"
    })).json()

len(data["d"]["Lokacije"]), len(data["d"]["Ugovori"])